# AAI 510 — Assignment 2
#### Mostafa Zamaniturk

## Exploratory Data Analysis with the Data Science Agent

**In this lab you will:**
- **Required:** Use the Databricks **Data Science Agent** (Agent Mode in the Assistant) to perform exploratory data analysis on the UltraFeedback dataset you loaded in Assignment 1.
- **Required:** Review the agent's output, identify what the data contains, and assess how it could be integrated into an agent (e.g., as tools, reference data, or evaluation benchmarks).
- **Required:** Provide your own written analysis of the data — the agent helps you explore, but **your interpretation is the deliverable**.

*This assignment is intentionally bare-bones. The notebook sets up the data reference; you drive the exploration through the Agent and record your findings below.*


---
## 1. About this assignment

Last week you loaded the **UltraFeedback** dataset into Unity Catalog as `main.default.assignment_file`. This week you'll explore that data in depth.

Instead of writing all EDA code by hand, you'll use the **Databricks Data Science Agent** — an AI assistant built into your notebook that can plan, generate, and run code on your behalf. Your job is to **guide the agent with good prompts**, **review what it produces**, and **write your own analysis** of the results.

### Why this matters

In agentic AI, the human-in-the-loop is critical. An agent can automate mechanical tasks (generating plots, computing statistics), but **you** are responsible for interpreting results, catching errors, and making decisions. This assignment practices that skill: you'll delegate the exploration to an agent, then own the analysis.

### How to use the Data Science Agent

1. Open the **Assistant** side panel in this notebook (click the Assistant icon or press **Cmd+I** / **Ctrl+I**).
2. In the bottom-right corner of the panel, toggle to **Agent** mode.
3. Type a prompt. Reference the table with `@main.default.assignment_file` so the agent knows which data to use.
4. The agent will create a plan, ask clarifying questions, and generate notebook cells. **Review each step** before clicking **Allow** or **Continue**.
5. After the agent finishes, read the generated cells and outputs. Add your own markdown cells with your interpretation.

**Docs:** [Use the Data Science Agent](https://docs.databricks.com/aws/en/notebooks/ds-agent)

**Requirements:**
- Partner-powered AI features must be enabled for your workspace.
- Databricks Assistant Agent Mode preview must be enabled. See [Manage Databricks previews](https://docs.databricks.com/aws/en/admin/workspace-settings/manage-previews).


---
## 2. Verify the dataset *(Required)*

Run the cell below to confirm the table from Assignment 1 is still available and to see a quick preview. If you get an error, re-run Assignment 1 first.


In [0]:
# Quick check: confirm the table exists and preview the first few rows
df = spark.table("main.default.assignment_file")

print(f"Row count: {df.count()}")
print(f"Columns: {df.columns}")

df.printSchema()
display(df.limit(5))

---
## 3. Agent-assisted EDA *(Required)*

Use the Data Science Agent to explore the dataset. Below are **sample prompts** to get you started — you don't need to use all of them, and you're encouraged to ask your own follow-up questions. The agent will generate code cells and outputs directly in this notebook.

### Sample prompts to try

**Data overview and quality:**
- "Describe the `@main.default.assignment_file` dataset. Show column statistics (counts, nulls, unique values) and data types. Think like a data scientist."
- "Check for missing values in every column of `@main.default.assignment_file`. Show a summary table and a heatmap of nulls."
- "How many duplicate rows are in `@main.default.assignment_file`? Show me examples if any exist."

**Distribution and patterns:**
- "What are the unique values in the `source` column of `@main.default.assignment_file`? Show a bar chart of how many rows come from each source."
- "Compare the average length (in characters) of the `chosen` vs `rejected` responses in `@main.default.assignment_file`. Visualize the distributions."
- "Which models appear most frequently as `chosen-model` and `rejected-model`? Show the top 10 of each."

**Content and usefulness for agent integration:**
- "Show me 5 example rows from `@main.default.assignment_file` where the `source` is 'evol_instruct'. What kind of instructions are these?"
- "Is there a relationship between the `source` column and which model was chosen? Create a cross-tabulation."
- "Which columns contain structured data (e.g., model names, sources) that could be turned into lookup or filtering tools for an agent?"
- "Identify any columns that would need transformation before being used in an agent workflow (e.g., text that needs parsing, inconsistent formats, columns that aren't useful)."

**Go deeper (optional):**
- "Create a word cloud of the most common terms in the instruction/prompt column."
- "Are there any outliers in response length? Show a box plot."
- "Summarize the key characteristics of this dataset in a new markdown cell."

> **Tip:** After the agent generates cells, read the output carefully. If something looks wrong or surprising, ask the agent to investigate further. This back-and-forth is how real agent-assisted analysis works.


*The Data Science Agent will insert cells below as you interact with it. Leave this section open for agent-generated content.*


In [0]:
# Load the dataset
df = spark.table("main.default.assignment_file")

# Basic statistics
print("=" * 80)
print("DATASET OVERVIEW")
print("=" * 80)
print(f"Total Rows: {df.count():,}")
print(f"Total Columns: {len(df.columns)}")
print(f"\nColumns: {', '.join(df.columns)}")
print("\n" + "=" * 80)
print("SCHEMA DETAILS")
print("=" * 80)
df.printSchema()

In [0]:
from pyspark.sql.functions import col, count, when
import pandas as pd

# Calculate null/missing counts for each column
print("=" * 80)
print("MISSING VALUES ANALYSIS")
print("=" * 80)

total_rows = df.count()

# Define which columns are numeric vs string
string_cols = ['source', 'prompt', 'chosen', 'chosen-model', 'rejected', 'rejected-model']
numeric_cols = ['chosen-rating', 'rejected-rating']

# Create expressions to count nulls - handle string and numeric columns differently
null_expressions = []
for c in df.columns:
    if c in string_cols:
        # For strings, check for null or empty string
        null_expressions.append(count(when(col(c).isNull() | (col(c) == ""), c)).alias(c))
    else:
        # For numeric, only check for null
        null_expressions.append(count(when(col(c).isNull(), c)).alias(c))

null_counts = df.select(null_expressions).collect()[0]

# Create a summary DataFrame
missing_data = []
for column in df.columns:
    null_count = null_counts[column]
    null_percentage = (null_count / total_rows) * 100
    missing_data.append({
        'Column': column,
        'Null Count': null_count,
        'Null %': f"{null_percentage:.2f}%",
        'Non-Null Count': total_rows - null_count
    })

missing_df = pd.DataFrame(missing_data)
print(missing_df.to_string(index=False))
print("\n" + "=" * 80)

In [0]:
from pyspark.sql.functions import col, countDistinct, approx_count_distinct

print("=" * 80)
print("CATEGORICAL COLUMNS - UNIQUE VALUE COUNTS")
print("=" * 80)

# String columns to analyze
string_columns = ['source', 'chosen-model', 'rejected-model']

for column in string_columns:
    unique_count = df.select(countDistinct(col(column))).collect()[0][0]
    print(f"\n{column.upper()}:")
    print(f"  Unique values: {unique_count}")
    
    # Show top 10 most frequent values
    print(f"\n  Top 10 most frequent values:")
    top_values = df.groupBy(column).count().orderBy(col("count").desc()).limit(10)
    display(top_values)

In [0]:
from pyspark.sql.functions import col, mean, stddev, min, max, percentile_approx

print("=" * 80)
print("NUMERIC COLUMNS - STATISTICAL SUMMARY")
print("=" * 80)

# Numeric columns
numeric_columns = ['chosen-rating', 'rejected-rating']

for column in numeric_columns:
    print(f"\n{column.upper()}:")
    stats = df.select(
        count(col(column)).alias('count'),
        mean(col(column)).alias('mean'),
        stddev(col(column)).alias('std_dev'),
        min(col(column)).alias('min'),
        max(col(column)).alias('max')
    ).collect()[0]
    
    print(f"  Count: {stats['count']:,}")
    print(f"  Mean: {stats['mean']:.4f}")
    print(f"  Std Dev: {stats['std_dev']:.4f}")
    print(f"  Min: {stats['min']:.4f}")
    print(f"  Max: {stats['max']:.4f}")

# Display distribution
print("\n" + "=" * 80)
print("RATING DISTRIBUTIONS")
print("=" * 80)
rating_stats = df.select('chosen-rating', 'rejected-rating')
display(rating_stats.summary())

In [0]:
from pyspark.sql.functions import length, col, mean, min, max

print("=" * 80)
print("TEXT COLUMNS - LENGTH STATISTICS")
print("=" * 80)

# Text columns to analyze
text_columns = ['prompt', 'chosen', 'rejected']

for column in text_columns:
    print(f"\n{column.upper()}:")
    
    # Calculate length statistics
    length_stats = df.select(
        mean(length(col(column))).alias('avg_length'),
        min(length(col(column))).alias('min_length'),
        max(length(col(column))).alias('max_length')
    ).collect()[0]
    
    print(f"  Average Length: {length_stats['avg_length']:.0f} characters")
    print(f"  Min Length: {length_stats['min_length']} characters")
    print(f"  Max Length: {length_stats['max_length']} characters")

In [0]:
print("=" * 80)
print("SAMPLE RECORDS (First 5 rows)")
print("=" * 80)
print("\nNote: Text fields may be truncated in display for readability\n")

display(df.limit(5))

In [0]:
# Install wordcloud library
%pip install wordcloud --quiet

## Columns Requiring Transformation for Agent Workflows

### ✅ **Ready to Use (No Transformation Needed)**

#### 1. **`source`** (Categorical)
* **Status:** Clean, 8 enumerated values
* **Use case:** Direct filtering, routing, stratification
* **Action:** None needed

#### 2. **`chosen-model` / `rejected-model`** (Categorical)
* **Status:** Clean, 17 model names
* **Use case:** Model lookup, filtering, comparison
* **Action:** None needed

#### 3. **`chosen-rating` / `rejected-rating`** (Numeric)
* **Status:** Clean numeric scores (1.0-5.0)
* **Use case:** Quality filtering, thresholding
* **Action:** None needed

---

### ⚠️ **Needs Transformation**

#### 4. **`chosen` / `rejected`** (Structured JSON Text)
* **Issue:** Contains JSON arrays representing conversation turns, not plain text
* **Structure:** `[{'content': '...', 'role': 'user'}, {'content': '...', 'role': 'assistant'}]`
* **Problems for agent workflows:**
  - Cannot directly use for Vector Search without extracting content
  - Role metadata (user/assistant) needs to be preserved or stripped depending on use case
  - Multi-turn conversations need to be flattened or structured appropriately
  - Raw JSON strings are not semantically searchable

* **Recommended Transformations:**
  1. **Parse JSON**: Convert string to structured array
  2. **Extract content**: Create new column `chosen_text_only` with concatenated content from all turns
  3. **Preserve structure**: Optionally keep conversation flow (user → assistant) for context-aware retrieval
  4. **For Vector Search**: Index the extracted plain text content, not the raw JSON
  
* **Example transformation (SQL/Python):**
  ```python
  from pyspark.sql.functions import from_json, col, concat_ws
  from pyspark.sql.types import ArrayType, StructType, StructField, StringType
  
  # Define schema
  message_schema = ArrayType(StructType([
      StructField('content', StringType(), True),
      StructField('role', StringType(), True)
  ]))
  
  # Parse and extract
  df_transformed = df.withColumn(
      'chosen_parsed', from_json(col('chosen'), message_schema)
  ).withColumn(
      'chosen_text_only', 
      concat_ws(' ', col('chosen_parsed.content'))
  )
  ```

---

#### 5. **`prompt`** (Unstructured Text)
* **Issue:** Natural language instructions, high variability
* **Status:** Clean text, but unstructured
* **Problems for agent workflows:**
  - No direct exact-match lookup possible
  - Cannot filter by semantic meaning without embedding
  - Length varies widely (7-14,512 chars)

* **Recommended Transformations:**
  1. **Vector Search indexing**: Embed using sentence transformers for semantic similarity
  2. **Length bucketing**: Optionally categorize by length for task complexity estimation
  3. **Keyword extraction**: Extract domain-specific terms for metadata filtering
  
* **Use case:** Semantic search to find similar historical prompts

---

### ❌ **Columns Not Useful for Agent Workflows (Consider Dropping)**

**(None identified)** — All columns have potential value:
- Structured columns (source, models, ratings) → filtering/lookup tools
- Text columns (prompt, chosen, rejected) → semantic search and retrieval

---

### **Priority Transformation Roadmap**

**Phase 1 (Immediate - for basic filtering tools):**
- Use structured columns as-is (source, models, ratings)
- Create Unity Catalog functions for filtering and lookup

**Phase 2 (Near-term - for semantic search):**
- Parse `chosen`/`rejected` JSON to extract plain text content
- Create new columns: `chosen_text`, `rejected_text`, `prompt_text`
- Index these text columns in Vector Search

**Phase 3 (Advanced - for context-aware retrieval):**
- Preserve conversation structure from `chosen`/`rejected`
- Build multi-turn conversation retrieval tools
- Add metadata extraction (prompt length, turn count, etc.)

---

### **Key Insight**

The **most critical transformation** is parsing the JSON structure in `chosen` and `rejected` columns. Without this, agents cannot:
- Perform semantic search on response content
- Extract actual text for RAG pipelines
- Use responses as templates for prompt engineering

All other columns are immediately usable for structured filtering and lookup operations.

In [0]:
from pyspark.sql.functions import col

print("=" * 80)
print("EVOL_INSTRUCT SOURCE - SAMPLE EXAMPLES")
print("=" * 80)

# Filter for evol_instruct source
evol_instruct_df = df.filter(col('source') == 'evol_instruct')

print(f"\nTotal evol_instruct examples: {evol_instruct_df.count():,}")
print("\nShowing 5 random examples:\n")

# Get 5 random samples
samples = evol_instruct_df.sample(fraction=0.01).limit(5).select(
    'prompt', 'chosen', 'chosen-model', 'chosen-rating', 'rejected-model', 'rejected-rating'
)

display(samples)

# Let's also look at the prompts more closely
print("\n" + "=" * 80)
print("DETAILED PROMPT EXAMPLES")
print("=" * 80)

for idx, row in enumerate(samples.collect(), 1):
    print(f"\n--- Example {idx} ---")
    print(f"Prompt: {row['prompt'][:500]}...") if len(row['prompt']) > 500 else print(f"Prompt: {row['prompt']}")
    print(f"\nChosen Model: {row['chosen-model']} (Rating: {row['chosen-rating']})")
    print(f"Rejected Model: {row['rejected-model']} (Rating: {row['rejected-rating']})")
    print(f"\nChosen Response Preview: {row['chosen'][:300]}...") if len(row['chosen']) > 300 else print(f"Chosen Response: {row['chosen']}")
    print("\n" + "-" * 80)

In [0]:
from pyspark.sql.functions import col, count
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

print("=" * 80)
print("SOURCE vs CHOSEN MODEL CROSS-TABULATION")
print("=" * 80)

# Create cross-tabulation
cross_tab = df.groupBy('source', 'chosen-model').count().orderBy('source', col('count').desc())

# Get top 3 models per source
print("\nTop 3 Most Chosen Models by Source:\n")
for source in df.select('source').distinct().collect():
    source_name = source['source']
    top_models = cross_tab.filter(col('source') == source_name).limit(3).toPandas()
    print(f"\n{source_name}:")
    for idx, row in top_models.iterrows():
        percentage = (row['count'] / df.filter(col('source') == source_name).count()) * 100
        print(f"  {idx+1}. {row['chosen-model']}: {row['count']:,} ({percentage:.1f}%)")

print("\n" + "=" * 80)
print("DETAILED CROSS-TABULATION TABLE")
print("=" * 80)

# Pivot table for visualization
pivot_data = cross_tab.toPandas().pivot_table(
    index='source', 
    columns='chosen-model', 
    values='count', 
    fill_value=0
)

print("\nFull Cross-Tabulation (showing counts):")
print(pivot_data.to_string())

# Calculate percentages by source
pivot_pct = pivot_data.div(pivot_data.sum(axis=1), axis=0) * 100

print("\n" + "=" * 80)
print("Percentage of Each Model per Source (%)")
print("=" * 80)
print(pivot_pct.round(1).to_string())

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Get top 8 most common models overall
top_models = df.groupBy('chosen-model').count().orderBy(col('count').desc()).limit(8).toPandas()['chosen-model'].tolist()

# Filter pivot data to top models only
pivot_filtered = pivot_pct[top_models]

# Create heatmap
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Heatmap 1: Percentage distribution
sns.heatmap(pivot_filtered, annot=True, fmt='.1f', cmap='YlGnBu', 
            cbar_kws={'label': 'Percentage (%)'}, ax=axes[0])
axes[0].set_title('Model Preference by Source (% within each source)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Chosen Model', fontsize=12)
axes[0].set_ylabel('Data Source', fontsize=12)
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')

# Grouped bar chart: Top 3 models per source
top_3_per_source = []
for source in pivot_filtered.index:
    top_3 = pivot_filtered.loc[source].nlargest(3)
    for model, pct in top_3.items():
        top_3_per_source.append({'Source': source, 'Model': model, 'Percentage': pct})

df_top3 = pd.DataFrame(top_3_per_source)
pivot_top3 = df_top3.pivot(index='Source', columns='Model', values='Percentage').fillna(0)

pivot_top3.plot(kind='bar', ax=axes[1], width=0.8, colormap='Set2')
axes[1].set_title('Top 3 Models per Source (% within source)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Data Source', fontsize=12)
axes[1].set_ylabel('Percentage (%)', fontsize=12)
axes[1].legend(title='Model', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("KEY INSIGHTS:")
print("=" * 80)

In [0]:
from pyspark.sql.functions import col, count, approx_count_distinct, rank
from pyspark.sql.window import Window
import pandas as pd

print("=" * 80)
print("STATISTICAL ANALYSIS: Source-Model Relationship")
print("=" * 80)

# Calculate diversity: How many different models are chosen per source?
print("\n1. MODEL DIVERSITY BY SOURCE:")
print("   (How many unique models are chosen from each source?)\n")

model_diversity = df.groupBy('source').agg(
    count('chosen-model').alias('total_choices'),
    approx_count_distinct('chosen-model').alias('unique_models')
).orderBy('unique_models', ascending=False).toPandas()

for idx, row in model_diversity.iterrows():
    diversity_ratio = row['unique_models'] / row['total_choices'] * 100
    print(f"   {row['source']:20s}: {row['unique_models']} unique models from {row['total_choices']:,} choices")

# Identify source-model affinity
print("\n" + "=" * 80)
print("2. STRONGEST SOURCE-MODEL AFFINITIES:")
print("   (Which source-model combinations are most common?)\n")

window_spec = Window.partitionBy('source')
affinity = df.groupBy('source', 'chosen-model').count().withColumn(
    'source_total', count('*').over(window_spec)
).withColumn(
    'percentage', (col('count') / col('source_total') * 100)
).orderBy('count', ascending=False).limit(15).toPandas()

for idx, row in affinity.iterrows():
    print(f"   {idx+1:2d}. {row['source']:20s} + {row['chosen-model']:20s}: {row['count']:5,} times ({row['percentage']:.1f}% of {row['source']} source)")

print("\n" + "=" * 80)
print("3. DOMINANT MODEL PER SOURCE:")
print("   (Which model is chosen most frequently for each source?)\n")

window_spec_rank = Window.partitionBy('source').orderBy(col('count').desc())
dominant = df.groupBy('source', 'chosen-model').count().withColumn(
    'rank', rank().over(window_spec_rank)
).filter(col('rank') == 1).orderBy('source').toPandas()

for idx, row in dominant.iterrows():
    total_for_source = df.filter(col('source') == row['source']).count()
    percentage = (row['count'] / total_for_source) * 100
    print(f"   {row['source']:20s} → {row['chosen-model']:20s} ({row['count']:,}/{total_for_source:,} = {percentage:.1f}%)")

print("\n" + "=" * 80)
print("CONCLUSION:")
print("There IS a relationship between source and chosen model.")
print("Different sources show varying model preferences, with GPT models")
print("dominating most sources but to different degrees.")
print("=" * 80)

In [0]:
from pyspark.sql.functions import col, count

print("=" * 80)
print("DUPLICATE ROW ANALYSIS")
print("=" * 80)

# Total number of rows
total_rows = df.count()
print(f"\nTotal rows in dataset: {total_rows:,}")

# Count distinct rows across all columns
distinct_rows = df.distinct().count()
print(f"Distinct rows: {distinct_rows:,}")

# Calculate duplicates
num_duplicates = total_rows - distinct_rows
print(f"\nNumber of duplicate rows: {num_duplicates:,}")

if num_duplicates > 0:
    print(f"Percentage of duplicates: {(num_duplicates / total_rows * 100):.2f}%")
    
    # Find rows that appear more than once
    print("\n" + "=" * 80)
    print("IDENTIFYING DUPLICATE RECORDS")
    print("=" * 80)
    
    # Group by all columns and count occurrences
    duplicate_check = df.groupBy(*df.columns).count().filter(col("count") > 1).orderBy(col("count").desc())
    
    duplicate_records = duplicate_check.count()
    print(f"\nNumber of unique rows that have duplicates: {duplicate_records:,}")
    
    if duplicate_records > 0:
        print("\nTop 5 most frequently duplicated records (showing count):")
        display(duplicate_check.select("source", "prompt", "chosen-model", "rejected-model", "count").limit(5))
        
        # Show full examples of duplicated rows
        print("\n" + "=" * 80)
        print("EXAMPLE DUPLICATE RECORDS")
        print("=" * 80)
        
        # Get one example of a duplicated row
        first_duplicate = duplicate_check.select(*[c for c in df.columns]).first()
        if first_duplicate:
            # Filter to show all occurrences of this duplicate
            example_filter = None
            for col_name in df.columns:
                col_val = first_duplicate[col_name]
                if example_filter is None:
                    example_filter = (col(col_name) == col_val)
                else:
                    example_filter = example_filter & (col(col_name) == col_val)
            
            duplicate_examples = df.filter(example_filter)
            print(f"\nShowing all {duplicate_examples.count()} occurrences of the first duplicate:")
            display(duplicate_examples)
else:
    print("\n✓ No duplicate rows found. All rows are unique.")
    print("\n" + "=" * 80)

In [0]:
from pyspark.sql.functions import col, count

print("=" * 80)
print("PARTIAL DUPLICATE ANALYSIS - Same Prompt")
print("=" * 80)

# Check for rows with the same prompt but potentially different responses
prompt_duplicates = df.groupBy("prompt").count().filter(col("count") > 1).orderBy(col("count").desc())

prompt_dup_count = prompt_duplicates.count()
total_dup_prompt_rows = prompt_duplicates.agg({"count": "sum"}).collect()[0][0]

print(f"\nPrompts that appear multiple times: {prompt_dup_count:,}")
print(f"Total rows with duplicate prompts: {total_dup_prompt_rows:,}")

if prompt_dup_count > 0:
    print("\nTop 5 most frequently repeated prompts:")
    display(prompt_duplicates.limit(5))
    
    # Show an example of a prompt with multiple responses
    most_common_prompt = prompt_duplicates.first()["prompt"]
    print("\n" + "=" * 80)
    print("EXAMPLE: Same prompt with different model responses")
    print("=" * 80)
    print(f"\nShowing all variations for the most frequently repeated prompt:")
    
    same_prompt_examples = df.filter(col("prompt") == most_common_prompt).select(
        "source", "prompt", "chosen-model", "chosen-rating", "rejected-model", "rejected-rating"
    )
    display(same_prompt_examples)
else:
    print("\n✓ No duplicate prompts found. All prompts are unique.")

print("\n" + "=" * 80)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns
import builtins  # Import to access Python's built-in min function

# Create comprehensive box plot visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Side-by-side box plots
ax1 = axes[0, 0]
box_data = [length_sample['chosen_length'], length_sample['rejected_length']]
box = ax1.boxplot(box_data, labels=['Chosen', 'Rejected'], patch_artist=True)
box['boxes'][0].set_facecolor('lightgreen')
box['boxes'][1].set_facecolor('lightcoral')
ax1.set_ylabel('Response Length (characters)', fontsize=11)
ax1.set_title('Box Plot: Response Length Distribution', fontsize=13, fontweight='bold')
ax1.grid(axis='y', alpha=0.3)

# Plot 2: Box plot by source (Chosen)
ax2 = axes[0, 1]
sns.boxplot(data=length_sample, y='chosen_length', x='source', ax=ax2, palette='Set2')
ax2.set_xticklabels(ax2.get_xticklabels(), rotation=45, ha='right', fontsize=9)
ax2.set_ylabel('Chosen Length (chars)', fontsize=10)
ax2.set_xlabel('Source', fontsize=10)
ax2.set_title('Chosen Response Length by Source', fontsize=12, fontweight='bold')
ax2.grid(axis='y', alpha=0.3)

# Plot 3: Histogram with outliers highlighted
ax3 = axes[1, 0]
ax3.hist(length_sample['chosen_length'], bins=50, alpha=0.7, color='green', edgecolor='black', label='All data')
ax3.axvline(chosen_upper, color='red', linestyle='--', linewidth=2, label=f'Upper threshold ({chosen_upper:.0f})')
ax3.axvline(chosen_lower, color='red', linestyle='--', linewidth=2, label=f'Lower threshold ({chosen_lower:.0f})')
ax3.set_xlabel('Chosen Response Length (characters)', fontsize=11)
ax3.set_ylabel('Frequency', fontsize=11)
ax3.set_title('Chosen Response Length Distribution with Outlier Thresholds', fontsize=12, fontweight='bold')
ax3.legend(fontsize=9)
ax3.grid(axis='y', alpha=0.3)

# Plot 4: Scatter plot - Chosen vs Rejected length
ax4 = axes[1, 1]
# Use Python's built-in min to avoid collision with Spark's min function
sample_scatter = length_sample.sample(n=builtins.min(1000, len(length_sample)))  # Limit for clarity
ax4.scatter(sample_scatter['chosen_length'], sample_scatter['rejected_length'], 
           alpha=0.3, s=10, color='blue')
ax4.axhline(rejected_upper, color='red', linestyle='--', alpha=0.5, label='Rejected upper threshold')
ax4.axvline(chosen_upper, color='red', linestyle='--', alpha=0.5, label='Chosen upper threshold')
ax4.set_xlabel('Chosen Length (chars)', fontsize=11)
ax4.set_ylabel('Rejected Length (chars)', fontsize=11)
ax4.set_title('Chosen vs Rejected Response Lengths', fontsize=12, fontweight='bold')
ax4.legend(fontsize=9)
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "=" * 80)
print("BOX PLOT INTERPRETATION:")
print("=" * 80)
print("\nThe box shows the interquartile range (IQR, 25th to 75th percentile).")
print("The line inside the box is the median (50th percentile).")
print("Whiskers extend to 1.5 × IQR from the box edges.")
print("Points beyond the whiskers are considered outliers.")
print("=" * 80)

In [0]:
print("=" * 80)
print("EXAMPLES OF OUTLIER RESPONSES")
print("=" * 80)

# Get actual outlier examples from the full dataset
print("\nSampling extreme outliers (very long responses)...\n")

# Very long chosen responses
long_chosen = df_lengths.filter(col('chosen_length') > chosen_upper).orderBy(col('chosen_length').desc()).limit(3).toPandas()

print("TOP 3 LONGEST CHOSEN RESPONSES:")
print("=" * 80)
for idx, row in long_chosen.iterrows():
    print(f"\nExample {idx+1}:")
    print(f"  Source: {row['source']}")
    print(f"  Model: {row['chosen-model']}")
    print(f"  Length: {row['chosen_length']:,} characters")
    print(f"  Prompt (first 150 chars): {row['prompt'][:150]}...")
    print("-" * 80)

# Very long rejected responses
long_rejected = df_lengths.filter(col('rejected_length') > rejected_upper).orderBy(col('rejected_length').desc()).limit(3).toPandas()

print("\n" + "=" * 80)
print("TOP 3 LONGEST REJECTED RESPONSES:")
print("=" * 80)
for idx, row in long_rejected.iterrows():
    print(f"\nExample {idx+1}:")
    print(f"  Source: {row['source']}")
    print(f"  Model: {row['rejected-model']}")
    print(f"  Length: {row['rejected_length']:,} characters")
    print(f"  Prompt (first 150 chars): {row['prompt'][:150]}...")
    print("-" * 80)

# Very short responses (if any)
short_chosen = df_lengths.filter(col('chosen_length') < chosen_lower).orderBy(col('chosen_length')).limit(3).toPandas()

if len(short_chosen) > 0:
    print("\n" + "=" * 80)
    print("UNUSUALLY SHORT CHOSEN RESPONSES:")
    print("=" * 80)
    for idx, row in short_chosen.iterrows():
        print(f"\nExample {idx+1}:")
        print(f"  Source: {row['source']}")
        print(f"  Model: {row['chosen-model']}")
        print(f"  Length: {row['chosen_length']:,} characters")
        print(f"  Prompt (first 150 chars): {row['prompt'][:150]}...")
        print("-" * 80)
else:
    print("\n✓ No unusually short chosen responses found (all within expected range)")

print("\n" + "=" * 80)
print("KEY INSIGHTS:")
print("=" * 80)
print("1. Outliers exist primarily on the HIGH end (very long responses)")
print("2. Both chosen and rejected can have extreme lengths (>10,000 chars)")
print("3. Most outliers are still legitimate responses, not data errors")
print("4. Source and task complexity influence response length variance")
print("=" * 80)

In [0]:
from pyspark.sql.functions import length, col
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("=" * 80)
print("OUTLIER ANALYSIS: Response Lengths")
print("=" * 80)

# Calculate lengths
df_lengths = df.select(
    col('source'),
    col('prompt'),
    length('chosen').alias('chosen_length'),
    length('rejected').alias('rejected_length'),
    col('chosen-model'),
    col('rejected-model')
)

# Get statistics for outlier detection
chosen_stats = df_lengths.select('chosen_length').summary().toPandas()
rejected_stats = df_lengths.select('rejected_length').summary().toPandas()

print("\nCHOSEN RESPONSE LENGTH STATISTICS:")
print(chosen_stats.to_string(index=False))

print("\nREJECTED RESPONSE LENGTH STATISTICS:")
print(rejected_stats.to_string(index=False))

# Sample data for visualization (10% for performance)
print("\nSampling 10% of data for visualization...")
length_sample = df_lengths.sample(fraction=0.1).toPandas()

# Calculate IQR and outliers
chosen_q1 = length_sample['chosen_length'].quantile(0.25)
chosen_q3 = length_sample['chosen_length'].quantile(0.75)
chosen_iqr = chosen_q3 - chosen_q1
chosen_lower = chosen_q1 - 1.5 * chosen_iqr
chosen_upper = chosen_q3 + 1.5 * chosen_iqr

rejected_q1 = length_sample['rejected_length'].quantile(0.25)
rejected_q3 = length_sample['rejected_length'].quantile(0.75)
rejected_iqr = rejected_q3 - rejected_q1
rejected_lower = rejected_q1 - 1.5 * rejected_iqr
rejected_upper = rejected_q3 + 1.5 * rejected_iqr

print("\n" + "=" * 80)
print("OUTLIER THRESHOLDS (IQR Method):")
print("=" * 80)
print(f"\nChosen responses:")
print(f"  Q1: {chosen_q1:.0f}, Q3: {chosen_q3:.0f}, IQR: {chosen_iqr:.0f}")
print(f"  Lower bound: {chosen_lower:.0f} chars")
print(f"  Upper bound: {chosen_upper:.0f} chars")

print(f"\nRejected responses:")
print(f"  Q1: {rejected_q1:.0f}, Q3: {rejected_q3:.0f}, IQR: {rejected_iqr:.0f}")
print(f"  Lower bound: {rejected_lower:.0f} chars")
print(f"  Upper bound: {rejected_upper:.0f} chars")

# Count outliers
chosen_outliers = length_sample[(length_sample['chosen_length'] < chosen_lower) | (length_sample['chosen_length'] > chosen_upper)]
rejected_outliers = length_sample[(length_sample['rejected_length'] < rejected_lower) | (length_sample['rejected_length'] > rejected_upper)]

print(f"\nChosen response outliers: {len(chosen_outliers):,} ({len(chosen_outliers)/len(length_sample)*100:.1f}% of sample)")
print(f"Rejected response outliers: {len(rejected_outliers):,} ({len(rejected_outliers)/len(length_sample)*100:.1f}% of sample)")

print("\n" + "=" * 80)

In [0]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt
from pyspark.sql.functions import col, lower, regexp_replace
import re

print("=" * 80)
print("WORD CLOUD: Most Common Terms in Prompts")
print("=" * 80)

# Sample prompts for performance (10% sample should be sufficient)
print("\nSampling 10% of prompts for analysis...")
prompt_sample = df.select('prompt').sample(fraction=0.1).collect()

# Combine all prompts into one text
all_text = ' '.join([row['prompt'] for row in prompt_sample])

print(f"Total characters analyzed: {len(all_text):,}")
print(f"Number of prompts sampled: {len(prompt_sample):,}\n")

# Create word cloud
wordcloud = WordCloud(
    width=1600, 
    height=800,
    background_color='white',
    colormap='viridis',
    max_words=100,
    relative_scaling=0.5,
    min_font_size=10,
    stopwords=set(['the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for', 
                   'of', 'with', 'by', 'from', 'is', 'are', 'was', 'were', 'be', 'been',
                   'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'should',
                   'can', 'could', 'may', 'might', 'must', 'that', 'this', 'these', 'those',
                   'i', 'you', 'he', 'she', 'it', 'we', 'they', 'what', 'which', 'who',
                   'when', 'where', 'why', 'how', 'as', 'if', 'then', 'so', 'than',
                   'your', 'their', 'its', 'my', 'his', 'her', 'our'])
).generate(all_text)

# Display word cloud
fig, ax = plt.subplots(figsize=(20, 10))
ax.imshow(wordcloud, interpolation='bilinear')
ax.axis('off')
ax.set_title('Most Common Terms in Prompt/Instruction Column', fontsize=20, fontweight='bold', pad=20)
plt.tight_layout(pad=0)
plt.show()

print("\n" + "=" * 80)
print("TOP 20 MOST FREQUENT WORDS:")
print("=" * 80)

# Get word frequencies
word_freq = wordcloud.words_
top_20 = sorted(word_freq.items(), key=lambda x: x[1], reverse=True)[:20]

for idx, (word, freq) in enumerate(top_20, 1):
    print(f"{idx:2d}. {word:20s}: {freq:.4f}")

print("\n" + "=" * 80)

In [0]:
import ast
from pyspark.sql.functions import udf, col
from pyspark.sql.types import StringType, ArrayType, StructType, StructField

print("=" * 80)
print("DEMONSTRATION: Parsing Structured Response Columns")
print("=" * 80)

# Define UDF to parse Python literal string and extract text content
def extract_content_from_response(response_str):
    """Parse Python literal list of dicts and extract concatenated content."""
    try:
        # Parse the string representation of list
        messages = ast.literal_eval(response_str)
        if isinstance(messages, list):
            # Extract 'content' from each message and join
            contents = [msg.get('content', '') for msg in messages if isinstance(msg, dict)]
            return ' '.join(contents)
        return response_str
    except:
        # If parsing fails, return original
        return response_str

extract_content_udf = udf(extract_content_from_response, StringType())

# Apply transformation to sample
print("\nTransforming 3 sample rows...\n")
df_transformed = df.limit(3).withColumn(
    'chosen_text_only', extract_content_udf(col('chosen'))
).withColumn(
    'rejected_text_only', extract_content_udf(col('rejected'))
)

# Show results
for idx, row in enumerate(df_transformed.select('source', 'chosen', 'chosen_text_only').collect(), 1):
    print(f"Example {idx}:")
    print(f"Source: {row['source']}")
    print(f"\nOriginal 'chosen' (first 200 chars):\n{row['chosen'][:200]}...")
    print(f"\nExtracted text (first 200 chars):\n{row['chosen_text_only'][:200]}...")
    print(f"\nOriginal length: {len(row['chosen'])} chars")
    print(f"Extracted length: {len(row['chosen_text_only'])} chars")
    print("\n" + "-" * 80 + "\n")

print("=" * 80)
print("TRANSFORMATION SUMMARY:")
print("=" * 80)
print("✓ Successfully parsed Python literal syntax")
print("✓ Extracted plain text content from structured messages")
print("✓ Ready for Vector Search indexing")
print("✓ Removed JSON/dict formatting overhead")
print("=" * 80)

---
### Potential Agent Integration Strategies

Based on the EDA, here are concrete ways this dataset could enhance an agentic AI system:

#### 1. **Vector Search for Similar Examples**
* **Index the `prompt` column** to enable semantic search for similar instructions
* **Index `chosen` responses** to find high-quality example responses for RAG
* Use case: "Given a user query, find the 5 most similar prompts and their chosen responses"

#### 2. **Model Preference Tool (Unity Catalog Function)**
* Create a UC function: `get_best_model_for_source(source_name)` 
  * Returns the most frequently chosen model for that data source
  * Example: For "evol_instruct", returns GPT-4 or GPT-3.5-turbo
* Create a UC function: `get_model_performance_stats(model_name)`
  * Returns average ratings, selection frequency, typical response length

#### 3. **Quality Benchmark Tool**
* **Filter by rating thresholds** to create evaluation sets
* Example tool: `get_high_quality_examples(min_rating=4.5, source='sharegpt', limit=10)`
* Use chosen vs rejected pairs for preference tuning or evaluation

#### 4. **Source-Specific Prompting**
* Tool to retrieve prompts by source type:
  * `evol_instruct`: Complex reasoning tasks
  * `sharegpt`: Conversational/chat examples  
  * `flan_v2_cot`: Chain-of-thought reasoning
* Agent could adapt its behavior based on instruction type

#### 5. **Response Length Predictor**
* Build a simple heuristic tool: Given a prompt, estimate expected response length
* Based on: prompt length, source, and historical patterns
* Useful for token budget planning in multi-step agent workflows

---
## Dataset Characteristics Summary

### **Overview**
The [main.default.assignment_file](#table) table is a **preference dataset** containing 157,675 human-evaluated comparisons of AI model responses. Each row captures a single prompt with two competing responses ("chosen" vs "rejected") along with ratings and model identifiers.

### **Structure & Data Quality**
* **Dimensions**: 157,675 rows × 8 columns
* **Schema**: `source` (string), `prompt` (string), `chosen` (string), `chosen-rating` (double), `chosen-model` (string), `rejected` (string), `rejected-rating` (double), `rejected-model` (string)
* **Data completeness**: Zero nulls across all columns (100% complete)
* **Duplicates**: Zero exact duplicates, but 56,744 unique prompts appear multiple times (97.4% of dataset) — this is intentional for cross-model comparison purposes

---

### **Content Distribution**

#### **Data Sources** (8 unique)
* **sharegpt**: 49,729 rows (31.5%) — conversational exchanges
* **flan_v2_niv2**: 38,127 rows (24.2%) — Natural Instructions dataset
* **evol_instruct**: 25,518 rows (16.2%) — complex, multi-step instructions
* **ultrachat**: 24,523 rows (15.6%) — dialogue data
* **flan_v2_cot**: 7,638 rows (4.8%) — chain-of-thought reasoning
* **false_qa**: 6,154 rows (3.9%) — error detection tasks
* **flan_v2_p3**: 4,775 rows (3.0%) — Public Pool of Prompts
* **flan_v2_flan2021**: 1,211 rows (0.8%) — FLAN 2021 tasks

#### **Model Landscape** (17 unique models)
**Top Chosen Models** (preferred responses):
* gpt-3.5-turbo: 25,862 selections (16.4%)
* gpt-4: 24,111 selections (15.3%)
* llama-2-70b-chat: 13,294 selections (8.4%)

**Top Rejected Models** (non-preferred responses):
* alpaca-7b: 14,151 rejections (9.0%)
* falcon-40b-instruct: 13,355 rejections (8.5%)
* wizardlm-7b: 12,853 rejections (8.2%)

**Key Pattern**: Larger, more capable models (GPT family, 70B parameter models) dominate chosen responses, while smaller 7B models are frequently rejected.

---

### **Quality Metrics**

#### **Rating Distributions**
* **Chosen ratings**: Mean 4.64 ± 0.47 (range: 1.25–5.0)
* **Rejected ratings**: Mean 3.32 ± 1.09 (range: 1.0–4.75)
* **Separation**: Clear 1.3-point average difference demonstrates strong preference signals
* **Variability**: Chosen responses show much tighter clustering (σ=0.47 vs σ=1.09), indicating higher consensus on quality

#### **Response Lengths**
* **Prompts**: Average 653 characters (range: 7–14,512)
* **Chosen responses**: Average 2,198 characters (range: 93–17,335)
* **Rejected responses**: Average 1,817 characters (range: 90–19,399)
* **Insight**: Chosen responses are ~21% longer on average, suggesting more thorough, detailed answers are preferred
* **Outliers present**: Both chosen and rejected responses contain extreme length outliers beyond IQR boundaries

---

### **Cross-Source Model Preferences**

Source-model relationships reveal domain-specific affinities:
* **GPT-3.5/GPT-4 dominate** 6 out of 8 sources (12-18% share each)
* **flan_v2_cot exception**: Bard is #1 chosen model (13.2%)
* **flan_v2_p3 exception**: Llama-2-13b and Llama-2-70b preferred (11.4-11.6%)
* **Model diversity**: All sources utilize 16-17 unique models, showing consistent cross-model evaluation

---

### **Evol Instruct Characteristics**
Examples reveal high complexity:
* Multi-step reasoning (physics + LaTeX, data + SQL + analysis)
* Constraint-heavy instructions (specific formats, quality requirements)
* Domain-crossing tasks (code + cooking, biology + logic)
* GPT models strongly preferred for complex reasoning tasks

---

### **Content Analysis**
**Prompt keywords** (word cloud analysis of 15,865 samples):
* Top terms: "answer" (1.0), "given" (0.79), "question" (0.77), "task" (0.63), "use" (0.52)
* **Pattern**: Instruction-following dataset with structured "Given X, do Y" format

---

### **Critical Data Transformation Requirements**

#### **✅ Ready for Immediate Use** (no transformation needed)
* `source`, `chosen-model`, `rejected-model`: Clean categorical values, enumerable
* `chosen-rating`, `rejected-rating`: Numeric filters for quality thresholding

#### **⚠️ Requires Transformation** (highest priority)
**`chosen` and `rejected` columns**:
* **Current format**: Python literal strings representing conversation turns:
  ```python
  [{'content': '...', 'role': 'user'}, {'content': '...', 'role': 'assistant'}]
  ```
* **Issue**: Cannot be used for Vector Search or RAG without parsing
* **Required action**: Parse with `ast.literal_eval()` and extract plain text content
* **Impact**: Blocks semantic similarity search and retrieval workflows

**`prompt` column**:
* Natural language, high variability
* Requires embedding for semantic similarity search
* Ready for Vector Search indexing after no transformation

---

### **Recommended Integration Paths**

#### **Immediate** (Unity Catalog Functions)
* `get_examples_by_source(source, limit)` — filter by data source
* `get_model_performance(model_name)` — retrieve ratings and selection frequency
* `get_high_quality_examples(min_rating, source, model)` — quality-based filtering
* `compare_models(model1, model2)` — head-to-head comparison

#### **Near-term** (Vector Search)
1. Parse `chosen`/`rejected` to extract plain text content
2. Create new columns: `chosen_text`, `rejected_text`, `prompt_text`
3. Index in Vector Search for semantic similarity
4. Enable prompt-based retrieval ("find similar instructions")

#### **Advanced** (Multi-turn & Context-Aware)
* Preserve conversation structure from parsed responses
* Multi-turn conversation retrieval tools
* Source-aware model recommendation
* Quality-based evaluation set generation

---

### **Key Takeaways**
1. **High-quality preference data**: Strong rating separation (1.3 points) with low variance in chosen responses
2. **Model hierarchy evident**: Clear performance tiers (GPT > 70B Llama > 40B models > 7B models)
3. **Intentional design**: Duplicate prompts enable fair cross-model comparison
4. **Transformation blocker**: `chosen`/`rejected` columns require parsing before agent integration
5. **Immediate value**: Structured columns (`source`, models, ratings) can power filtering tools today

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Subplot 1: Top 10 Sources
source_counts = df.groupBy('source').count().orderBy('count', ascending=False).limit(10).toPandas()
axes[0, 0].barh(range(len(source_counts)), source_counts['count'], color='steelblue')
axes[0, 0].set_yticks(range(len(source_counts)))
axes[0, 0].set_yticklabels(source_counts['source'])
axes[0, 0].set_xlabel('Number of Examples')
axes[0, 0].set_title('Distribution of Data Sources')
axes[0, 0].invert_yaxis()

# Subplot 2: Top 10 Chosen Models
chosen_models = df.groupBy('chosen-model').count().orderBy('count', ascending=False).limit(10).toPandas()
axes[0, 1].barh(range(len(chosen_models)), chosen_models['count'], color='green')
axes[0, 1].set_yticks(range(len(chosen_models)))
axes[0, 1].set_yticklabels(chosen_models['chosen-model'])
axes[0, 1].set_xlabel('Number of Times Chosen')
axes[0, 1].set_title('Top 10 Most Frequently Chosen Models')
axes[0, 1].invert_yaxis()

# Subplot 3: Top 10 Rejected Models
rejected_models = df.groupBy('rejected-model').count().orderBy('count', ascending=False).limit(10).toPandas()
axes[1, 0].barh(range(len(rejected_models)), rejected_models['count'], color='red')
axes[1, 0].set_yticks(range(len(rejected_models)))
axes[1, 0].set_yticklabels(rejected_models['rejected-model'])
axes[1, 0].set_xlabel('Number of Times Rejected')
axes[1, 0].set_title('Top 10 Most Frequently Rejected Models')
axes[1, 0].invert_yaxis()

# Subplot 4: Response Length Distribution
length_stats = df.select(
    length('prompt').alias('Prompt'),
    length('chosen').alias('Chosen'),
    length('rejected').alias('Rejected')
).sample(0.1).toPandas()  # Sample for faster plotting

axes[1, 1].boxplot([length_stats['Prompt'], length_stats['Chosen'], length_stats['Rejected']], 
                    labels=['Prompt', 'Chosen', 'Rejected'])
axes[1, 1].set_ylabel('Character Count')
axes[1, 1].set_title('Text Length Distributions (Box Plot)')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n" + "="*80)
print("KEY OBSERVATIONS:")
print("="*80)
print("1. GPT-3.5 and GPT-4 dominate the 'chosen' category")
print("2. Smaller models (Alpaca-7B, WizardLM-7B) frequently rejected")
print("3. Chosen responses are typically longer than rejected ones")
print("4. Data is well-distributed across multiple sources")
print("="*80)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
plt.figure(figsize=(14, 5))

# Subplot 1: Rating Distributions Comparison
plt.subplot(1, 2, 1)
rating_data = df.select('chosen-rating', 'rejected-rating').toPandas()
sns.histplot(data=rating_data, x='chosen-rating', bins=20, alpha=0.6, label='Chosen', color='green')
sns.histplot(data=rating_data, x='rejected-rating', bins=20, alpha=0.6, label='Rejected', color='red')
plt.xlabel('Rating')
plt.ylabel('Frequency')
plt.title('Rating Distribution: Chosen vs Rejected Responses')
plt.legend()

# Subplot 2: Average Ratings by Source
plt.subplot(1, 2, 2)
avg_ratings = df.groupBy('source').agg(
    mean('chosen-rating').alias('Chosen'),
    mean('rejected-rating').alias('Rejected')
).toPandas().sort_values('Chosen', ascending=True)

x_pos = range(len(avg_ratings))
plt.barh(x_pos, avg_ratings['Chosen'], alpha=0.7, label='Chosen', color='green')
plt.barh(x_pos, avg_ratings['Rejected'], alpha=0.7, label='Rejected', color='red')
plt.yticks(x_pos, avg_ratings['source'])
plt.xlabel('Average Rating')
plt.ylabel('Source')
plt.title('Average Ratings by Data Source')
plt.legend()
plt.tight_layout()

plt.show()

print("\n" + "="*80)
print("Insight: Chosen responses consistently score ~1.3 points higher than rejected responses")
print("across all sources, showing clear preference signals in the dataset.")
print("="*80)

---
### Key Findings from EDA

**Dataset Overview:**
* **157,675 rows** with **8 columns**
* **No missing values** across any column (0% nulls)
* Clean, well-structured preference dataset

**Data Sources:**
* **8 unique sources**: ShareGPT (49,729), Flan V2 NIV2 (38,127), Evol Instruct (25,518), UltraChat (24,523), and 4 others
* Diverse instruction types from different data generation pipelines

**Model Coverage:**
* **17 unique models** in both chosen and rejected categories
* Top chosen models: GPT-3.5-Turbo (25,862), GPT-4 (24,111), Llama-2-70B (13,294)
* Top rejected models: Alpaca-7B (14,151), Falcon-40B (13,355), WizardLM-7B (12,853)
* Clear preference for larger, more capable models

**Rating Distributions:**
* **Chosen ratings**: Mean 4.64, Std 0.47 (range 1.25–5.0) — highly rated responses
* **Rejected ratings**: Mean 3.32, Std 1.09 (range 1.0–4.75) — more varied quality
* Clear separation between chosen (preferred) and rejected responses

**Text Content:**
* **Prompts**: Average 653 chars (range 7–14,512)
* **Chosen responses**: Average 2,198 chars (range 93–17,335) — slightly longer
* **Rejected responses**: Average 1,817 chars (range 90–19,399) — slightly shorter
* Substantial text data suitable for semantic analysis

---
## 4. Your analysis *(Required)*

After the agent has helped you explore the data, write your own analysis in the sections below. **This is the core deliverable** — the agent does the mechanical work; you provide the thinking. Do not use AI for this task.

### 4a. What does this dataset contain?

Summarize what you found. Include:
- How many rows and columns are there?
- What do the key columns represent (e.g., source, instruction, chosen, rejected, models)?
- What is the overall structure of a single record — what does one row "mean"?
- Any data quality issues (nulls, duplicates, inconsistencies)?

*Write your summary below (replace this text):*

**[Your answer here]**

My findings:

This dataset consists of 8 columns and 157,675 rows, with no null values. The source of prompts: the user's input/question injected into the AI model, the response judged to be better and its rating; the chosen model; the response judged to be worse or less preferable and its rating; and, lastly, the rejected model.

There are no duplication considering all columns, but there are partial duplication in the dataset, that are related to the prompts that were intentionally used again to evaluate different model results. This approach helped to compare models, find relative model preferences, and build comparative evaluation tools for agents. 

A single record in this dataset represents a distinct, face-to-face evaluation in which a user or script sends the same request from a specific source criterion to two separate AI models (selected model and rejected model), which then produce independent responses. A human or automated judge evaluates both outputs, scores them via the selected rating and rejected rating columns, and officially crowns one response as the preferred winner (selected) and the other as the inferior option (rejected).

There are 8 unique source values that represent the different training datasets (benchmarks). In model and source analysis, it illustrates the usage of different data sources, their percentage, and for the models, all the information about the types of models and the categorical analysis are shown for each model. Then, based on the chosen or rejected model, some visulizations were available to give better insights about different models. 

ShareGPT dominates about one-third of all examples, Flan variants make up nearly 33 percent for instruction-following tasks, Evol Instruct and UltraChat each had a 15%-16% contribution for complex reasoning and diverse dialogues.

In addition, response length comparisoned through prompt, chosen, and rejected ones, which can give insights about the average, minimum, and maximum length of the responses.

There are a lot of other information extracted from the dataset, such as the top-most frequently rejected models, the relationship between source and chosen model. 


### 4b. How could this data be integrated into an agent?

In this course you won't be training models directly — instead, you'll be building **agents** that leverage data through **tools** (Unity Catalog functions, Vector Search, MCP). Based on your EDA, assess how this dataset could be useful in an agent workflow:
- Which columns could an agent **query as a tool** (e.g., a function that looks up model preferences by source, or retrieves example prompts by category)?
- Could any text columns (instructions, chosen/rejected responses) be indexed for **Vector Search** so an agent can find similar examples?

*Write your assessment below (replace this text):*

**[Your answer here]**

This dataset can strongly help to choose between models and data sources to utilize for the desired agents. These EDA results help to figure out which model and data source work better in different demand. For example, if the agent task is to do complex reasoning and diverse dialogue, it is better to use Evol Instruct or UltraChat as a data sourse. And, for the model, using the model that was frequently marked as a better performance model can be used for a specific agent. This strongly help the architecture to make a harmonised model architecture for the agent. 



The best column for a filtering tool would be the "source" column to pull specific data types. In some cases, an agent needs organic human conversations, in some other cases, academic NLP tasks are needed for specific tasks.



And, based on the source, other important data will be used, the choosen model and the rating for it. In the context of chosen responses, the results demonstrate that they are 21% longer on average than rejected responses. Obviously, the prefered responses tend to be more comprehensive, have more quality, and are explained with more details. Other data sources also follow this pattern, which can be very helpful for building agents. Information above can be seen from the visualizations that the high-quality responses asd their length have a relationship. This reveal the vector's information to use for a specific agent's needs.



As I mentioned in the previous section, there is a strong relationship between source and chosen model. This is very important for agent integration. Look at it like an engine and gearbox, and a real steep land. The engine and gearbox separately are matters alot, but the compatibility of them can give us a desired result when going up the hills. The strong source and model affinities with the percentage are mentioned below.

1- ShareGPT + GPT-3.5-Turbo: 9,194 times (18.5% of ShareGPT)

2- ShareGPT + GPT-4: 8,705 times (17.5% of ShareGPT)

3- Flan V2 NIV2 + GPT-3.5-Turbo: 5,652 times (14.8%)

4- Flan V2 NIV2 + GPT-4: 5,203 times (13.6%)

5- Evol Instruct + GPT-3.5-Turbo: 4,675 times (18.3%)

Another useful piece of information is the highly structured columns. The first is the source, which is categorical with eight values. It can be used for filteringf function, recommendation function and statistics functions of an agent. 

Why it matters: "Source indicates instruction type/complexity. An agent could route queries based on whether the user needs conversational (ShareGPT), reasoning (evol_instruct), or chain-of-thought (flan_v2_cot) examples.”

Chosen-model, categorical with 17 different values. Useful for lookup function, filter function, and comparison functions. 

Why it matters: "Agents could use this to benchmark models, retrieve examples of specific model outputs, or recommend models for specific tasks based on historical performance."

Prompt column is a semi-structured column that require processing before any agent uses it. It can be useful for semantic similarity and example retrieval. and category detection. This column, while not directly structured, by indexing the prompts it can help for semantic search, which can find relevant examples for an agent.

Another interesting reveal from the dataset is the word cloud analysis. It shows the top 20 most frequent words and based on it, the dataset is heavily focused on the instreuctions and task-oriented language. The "answer", "given," and "question" illustrate the dominance of the instruction-following dataset, where prompts ask models to perform a specific task based on the provided information.


%md
### 4c. How well did the Data Science Agent perform?

Reflect on using the agent for EDA:
- What did the agent do well? Where did it save you time?
- Did the agent make any mistakes or produce anything you had to correct?
- What prompts worked best? What would you do differently next time?

*Write your reflection below (replace this text):*

**[Your answer here]**

The first significant thing about the agents is that they work based on our needs. The prompts that they recieve and the result that they present can help to put the next step in the path. The pivot is important, what we need and what can get us close to the result detemine out effort through it. The skills matter alot here, for someone who does not have experience or does not know about the steps to solving a problem is creating a much more bigger problem! But with the right skills it can help alot. For example, for writing codes, it definately save time, as well as after giving a result on visualization, it can help us to think about how to revise or how to ask the next question as a prompt.

Yes, absolutely. As the question be straigt forward, the agent produces better answers. By giving the agent several tasks at a time, the chance of making a mistake will increase. This is where we, as AI developers, need to have the skills to use the agents, writing goal-oriented and purpose-driven prompts with a touch of creativity.

The next time that I decide to use agents for EDA, I will try to make a plan before starting writing code. Actually, the steps of extracting the information through the traditional EDA methods can help alot. This time, based on the questions and prompts, I will make a list in order to reveal more useful information form any dataset.

Lastly, I am going to share one of my experiences using the agent. I try to use different agents from different companies. The reason behind it is that the models behind the agents works based on their trained data and the similarity in vectors. Using different agents can help to see a subject from different angles, so more "aha-moments" can happen by utilizing this method.

---
## Lab complete

**Required:**
- **Section 2:** The dataset verification cell ran and confirmed the table exists with rows and schema.
- **Section 3:** You used the Data Science Agent to explore the data. Agent-generated cells with outputs are visible in the notebook.
- **Section 4a:** You wrote a summary of what the dataset contains.
- **Section 4b:** You assessed how this data could be integrated into an agent (tools, Vector Search, evaluation).
- **Section 4c:** You reflected on the agent's performance and the human-in-the-loop experience.

**Submit:** Your executed notebook (`.ipynb` with all outputs, including agent-generated cells) and the completed `SUBMISSION_2.md`.

*Next week you'll create tools (Unity Catalog functions, Vector Search, MCP) for the agent you'll build over Weeks 3–5.*
